In [7]:
import yfinance as yf
import pandas as pd
import os

os.makedirs("data", exist_ok=True)

Download and flatten each ticker

In [8]:
def download_clean(ticker, col_name, start, end):
    df = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
    df = df[['Close']].copy()
    # Flatten multi-level columns if present
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [col_name]
    else:
        df.columns = [col_name]
    df.index.name = 'date'
    return df

start = "2010-01-01"
end = "2026-03-26"

Download all three

In [9]:
brent = download_clean("BZ=F", "brent_price", start, end)
wti   = download_clean("CL=F", "wti_price", start, end)
usd   = download_clean("DX-Y.NYB", "usd_index", start, end)

Save individual CSVs

In [10]:
brent.to_csv("data/brent_raw.csv")
wti.to_csv("data/wti_raw.csv")
usd.to_csv("data/usd_raw.csv")

Merge into master

In [11]:
master = brent.join(wti, how='left').join(usd, how='left')
master['brent_wti_spread'] = master['brent_price'] - master['wti_price']
master.to_csv("data/master_raw.csv")

print(master.tail(10))
print(f"\nShape: {master.shape}")
print(f"\nNulls:\n{master.isnull().sum()}")

            brent_price  wti_price   usd_index  brent_wti_spread
date                                                            
2026-03-12   100.459999  95.730003   99.739998          4.729996
2026-03-13   103.139999  98.709999  100.360001          4.430000
2026-03-16   100.209999  93.500000   99.709999          6.709999
2026-03-17   103.419998  96.209999   99.580002          7.209999
2026-03-18   107.379997  96.320000  100.089996         11.059998
2026-03-19   108.650002  96.139999   99.230003         12.510002
2026-03-20   112.190002  98.320000   99.650002         13.870003
2026-03-23    99.940002  88.129997   98.949997         11.810005
2026-03-24   104.489998  92.349998   99.430000         12.139999
2026-03-25   102.220001  90.320000   99.599998         11.900002

Shape: (4050, 4)

Nulls:
brent_price         0
wti_price           1
usd_index           2
brent_wti_spread    1
dtype: int64
